# Legacy Analysis Workbook Inspection

Inspect the existing Excel analysis artifact without treating it as an authoritative source. Reproducible results should be regenerated from the CSV exports.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd

def root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        q=p/'stack_exchange_analysis'
        if (p/'database').exists(): return p
        if (q/'database').exists(): return q
    raise FileNotFoundError
PROJECT_ROOT=root(); ANALYSIS_DIR=PROJECT_ROOT/'analysis'; sys.path.insert(0,str(PROJECT_ROOT/'src'))
from analysis_utils import parse_best_date_column
path=ANALYSIS_DIR/'cumulative-answers-questions-stackexchange.xlsx'
if not path.exists(): raise FileNotFoundError(path)
workbook=pd.ExcelFile(path); print('Sheets:',workbook.sheet_names)

## Sheet inventory

In [ ]:
frames={}; rows=[]
for sheet in workbook.sheet_names:
    frame=pd.read_excel(path,sheet_name=sheet); frames[sheet]=frame; rows.append({'sheet':sheet,'rows':len(frame),'columns':frame.shape[1],'missing_cells_pct':frame.isna().mean().mean()*100,'duplicate_rows':int(frame.duplicated().sum())})
inventory=pd.DataFrame(rows); display(inventory)

## Previews and inferred schema

In [ ]:
for sheet,frame in frames.items():
    print(f'\n=== {sheet} ==='); display(frame.head(10)); date_col,parsed=parse_best_date_column(frame); numeric=frame.select_dtypes(include=np.number).columns.tolist(); print({'candidate_date_column':date_col,'date_min':parsed.min() if parsed is not None else None,'date_max':parsed.max() if parsed is not None else None,'numeric_columns':numeric})

## Takeaways
Treat the workbook as a legacy presentation/analysis artifact. Use it for reconciliation only; regenerate substantive findings from source CSVs through the numbered notebooks.